In [1]:
packages = ["sklearn", "mlflow", "shap", "mlxtend"]

for pkg in packages:
    try:
        module = __import__(pkg)
        version = getattr(module, "__version__", "unknown version")
        print(f"{pkg}: installed ({version})")
    except ImportError:
        print(f"{pkg}: NOT installed")

sklearn: installed (1.7.2)
mlflow: NOT installed
shap: NOT installed
mlxtend: NOT installed


In [1]:
packages = ["sklearn", "mlflow", "shap", "mlxtend"]

for pkg in packages:
    try:
        module = __import__(pkg)
        version = getattr(module, "__version__", "unknown version")
        print(f"{pkg}: installed ({version})")
    except ImportError:
        print(f"{pkg}: NOT installed")

sklearn: installed (1.9.1)
mlflow: installed (3.16.0)
shap: installed (0.52.0)
mlxtend: installed (0.25.0)


### Part 3 Setup: Reuse Part 2's Pipeline and Feature Engineering

Part 3 builds directly on Part 2's cleaned, feature-engineered dataset rather than
reloading/recleaning the raw CSV independently — same reuse pattern as
`visualizations.py` and `traffic_cli.py` importing from `pipeline.py` and
`feature_engineering.py`, just now reaching one directory further up from
`part3_machine_learning/notebooks/`.

In [2]:
import sys
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # notebooks -> part3_machine_learning -> project root
PART2_DIR = PROJECT_ROOT / "part2_python"
sys.path.insert(0, str(PART2_DIR))

from pipeline import run_pipeline, DATA_FILE
from feature_engineering import engineer_features

clean_df = run_pipeline(DATA_FILE)
feat_df = engineer_features(clean_df)
feat_df.shape

Standardised casing to lowercase in 1730 rows of 'weather_description' (merging case-variant duplicates such as 'Sky is Clear' / 'sky is clear').
Removed 17 exact duplicate rows.
Collapsed 5430 duplicate-timestamp hours (7612 rows removed) down to one row per hour; 0 of these hours had inconsistent traffic_volume across duplicates.
Found 10 rows with physically impossible 'temp' readings (<= 0 Kelvin).
Found 1 rows with physically implausible 'rain_1h' readings (> 1000mm/hour): [9831.3]
Imputed 4 missing 'temp' value(s) in month 1 using that month's median (265.48).
Imputed 6 missing 'temp' value(s) in month 2 using that month's median (265.89).
Imputed 1 missing 'rain_1h' value(s) in month 7 using that month's median (0.00).


(40575, 29)

### Proxy Accident-Risk Label

No accident dataset exists, so we build a documented proxy `high_risk` label per the brief.

**`SEVERE_WEATHER`** reuses the exact same definition as Part 2's `is_severe_weather` flag
(`Thunderstorm`, `Squall`), for consistency across the whole project.

**`is_low_visibility`** is new: weather conditions that impair visibility —
`Fog`, `Mist`, `Haze`, `Smoke`.

**Congestion bucketing for this label uses 4 categories** (Low/Medium/High/Severe, split on
the 25th/50th/75th percentiles), which is different from Part 2's `congestion_category`
(3 categories, split on 25th/75th only). To avoid overwriting or confusing Part 2's column,
this one is named `risk_congestion_category` and lives only in Part 3.

A row is `high_risk` when High/Severe congestion coincides with severe or low-visibility
weather.

In [3]:
import numpy as np

SEVERE_WEATHER = ["Thunderstorm", "Squall"]
LOW_VISIBILITY_CONDITIONS = ["Fog", "Mist", "Haze", "Smoke"]

feat_df["is_low_visibility"] = feat_df["weather_main"].isin(LOW_VISIBILITY_CONDITIONS).astype(int)

# Congestion category (data-driven quartiles of traffic_volume) — 4 buckets, used only
# to construct the proxy risk label, distinct from Part 2's 3-bucket congestion_category
q1, q2, q3 = feat_df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values

def bucket(v):
    if v <= q1:
        return "Low"
    elif v <= q2:
        return "Medium"
    elif v <= q3:
        return "High"
    return "Severe"

feat_df["risk_congestion_category"] = feat_df["traffic_volume"].apply(bucket)

# Proxy accident-risk label
high_congestion = feat_df["risk_congestion_category"].isin(["High", "Severe"])
risky_weather = (
    feat_df["weather_main"].isin(SEVERE_WEATHER)
    | (feat_df["is_low_visibility"] == 1)
)
feat_df["high_risk"] = (high_congestion & risky_weather).astype(int)

print(feat_df["risk_congestion_category"].value_counts())
print()
print(feat_df["is_low_visibility"].value_counts())
print()
print(feat_df["high_risk"].value_counts())
print(feat_df["high_risk"].value_counts(normalize=True))

risk_congestion_category
High      10145
Medium    10144
Low       10144
Severe    10142
Name: count, dtype: int64

is_low_visibility
0    36524
1     4051
Name: count, dtype: int64

high_risk
0    38649
1     1926
Name: count, dtype: int64
high_risk
0    0.952532
1    0.047468
Name: proportion, dtype: float64


### Completing the Common Feature Set for Task 1

Two pieces the brief requires that we don't have yet:
- **Holiday flag**: `holiday` is currently NaN for non-holidays and a holiday's name (e.g. 'Labor Day') otherwise — not usable as a numeric model input directly. We convert it to a clean binary `is_holiday`.
- **Cyclical encoding of day of week**: Part 2 only cyclically encoded hour (`hour_sin`/`hour_cos`). Adding `dow_sin`/`dow_cos` the same way (period 7 instead of 24) captures that Sunday (6) and Monday (0) are adjacent, not far apart.

In [4]:
feat_df["is_holiday"] = feat_df["holiday"].notna().astype(int)

feat_df["dow_sin"] = np.sin(2 * np.pi * feat_df["day_of_week"] / 7)
feat_df["dow_cos"] = np.cos(2 * np.pi * feat_df["day_of_week"] / 7)

print(feat_df["is_holiday"].value_counts())
print(feat_df[["day_of_week", "dow_sin", "dow_cos"]].drop_duplicates().sort_values("day_of_week"))

is_holiday
0    40522
1       53
Name: count, dtype: int64
     day_of_week   dow_sin   dow_cos
126            0  0.000000  1.000000
0              1  0.781831  0.623490
15             2  0.974928 -0.222521
35             3  0.433884 -0.900969
59             4 -0.433884 -0.900969
81             5 -0.974928 -0.222521
104            6 -0.781831  0.623490


## Task 1 — Supervised Machine Learning Models

### Step 1: Load the Part 3 dataset and build a chronological train/test split

This cell replaces the earlier prototyping cells (proxy label construction, holiday/day of week encoding) with a single call to the now consolidated `data_prep.py`, so the notebook and the standalone script are always building from the exact same logic rather than two copies that could drift apart.

Because this is genuine hourly time series data spanning October 2012 to September 2018, we deliberately avoid a random train/test split. Shuffling would let a model train on rows that come chronologically after some of its own test points, information a real deployed model would never have at prediction time. Instead we sort by `date_time` and cut at a fixed point in the timeline:

- Everything before the cutoff date becomes the training set.
- Everything from the cutoff date onward becomes the test set.
- The cutoff sits at roughly 80% of the way through the full date range, so the training set is the past and the test set is the most recent slice, the way it would work in production.
- The same split is reused for both models in this task, the `high_risk` classifier and the `traffic_volume` regressor, so both are judged on the same held out period and results stay comparable.

In [5]:
import sys
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

PART3_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PART3_DIR))

from data_prep import prepare_data

ml_df = prepare_data()
ml_df = ml_df.sort_values("date_time").reset_index(drop=True)

split_idx = int(len(ml_df) * 0.8)
cutoff_date = ml_df.loc[split_idx, "date_time"]

train_df = ml_df[ml_df["date_time"] < cutoff_date].copy()
test_df = ml_df[ml_df["date_time"] >= cutoff_date].copy()

print(f"Cutoff date: {cutoff_date}")
print(f"Train: {train_df.shape[0]} rows  ({train_df['date_time'].min()} to {train_df['date_time'].max()})")
print(f"Test:  {test_df.shape[0]} rows  ({test_df['date_time'].min()} to {test_df['date_time'].max()})")
print(f"Train high_risk rate: {train_df['high_risk'].mean():.2%}")
print(f"Test  high_risk rate: {test_df['high_risk'].mean():.2%}")

Standardised casing to lowercase in 1730 rows of 'weather_description' (merging case-variant duplicates such as 'Sky is Clear' / 'sky is clear').
Removed 17 exact duplicate rows.
Collapsed 5430 duplicate-timestamp hours (7612 rows removed) down to one row per hour; 0 of these hours had inconsistent traffic_volume across duplicates.
Found 10 rows with physically impossible 'temp' readings (<= 0 Kelvin).
Found 1 rows with physically implausible 'rain_1h' readings (> 1000mm/hour): [9831.3]
Imputed 4 missing 'temp' value(s) in month 1 using that month's median (265.48).
Imputed 6 missing 'temp' value(s) in month 2 using that month's median (265.89).
Imputed 1 missing 'rain_1h' value(s) in month 7 using that month's median (0.00).


Cutoff date: 2017-10-26 18:00:00
Train: 32460 rows  (2012-10-02 09:00:00 to 2017-10-26 17:00:00)
Test:  8115 rows  (2017-10-26 18:00:00 to 2018-09-30 23:00:00)
Train high_risk rate: 4.59%
Test  high_risk rate: 5.37%


### Diagnostic: why does the test period have a higher high_risk rate?

The proxy label combines a congestion threshold (computed globally, across all years) with a severe or low-visibility weather flag. A higher `high_risk` rate in the test period could come from either component. Grouping by year isolates which one is responsible.

In [6]:
diagnostic = ml_df.copy()
diagnostic["year"] = diagnostic["date_time"].dt.year

yearly_summary = diagnostic.groupby("year").agg(
    avg_traffic_volume=("traffic_volume", "mean"),
    pct_high_or_severe_congestion=("risk_congestion_category", lambda s: s.isin(["High", "Severe"]).mean()),
    pct_severe_weather=("weather_main", lambda s: s.isin(["Thunderstorm", "Squall"]).mean()),
    pct_low_visibility=("is_low_visibility", "mean"),
    pct_high_risk=("high_risk", "mean"),
)

print(yearly_summary.round(4))

      avg_traffic_volume  pct_high_or_severe_congestion  pct_severe_weather  \
year                                                                          
2012           3226.7019                         0.4784              0.0019   
2013           3309.5528                         0.4936              0.0048   
2014           3270.1433                         0.4901              0.0029   
2015           3258.0420                         0.4940              0.0095   
2016           3193.6952                         0.4886              0.0079   
2017           3376.5891                         0.5205              0.0189   
2018           3323.9013                         0.5105              0.0216   

      pct_low_visibility  pct_high_risk  
year                                     
2012              0.1583         0.0737  
2013              0.0893         0.0404  
2014              0.0395         0.0182  
2015              0.1258         0.0509  
2016              0.0981         0.0

### Step 2: Define the feature set for Task 1 models

`high_risk` is derived from `risk_congestion_category` (built from `traffic_volume`) and weather conditions. To avoid leakage, `traffic_volume`, its scaled version, and both congestion category columns are excluded as predictors, since including them would let a model trivially reconstruct the label rather than learn a genuine pattern. Weather encodings and time-based features remain, since those represent information legitimately available at prediction time.

First, list every column in `ml_df` so the include/exclude list below is built against the actual schema, not assumption.

In [7]:
for col in ml_df.columns:
    print(f"{col:30s} {ml_df[col].dtype}")

holiday                        object
temp                           float64
rain_1h                        float64
snow_1h                        float64
clouds_all                     int64
weather_main                   object
weather_description            object
date_time                      datetime64[ns]
traffic_volume                 int64
hour                           int32
day_of_week                    int32
is_weekend                     int64
hour_sin                       float64
hour_cos                       float64
wx_Clear                       int64
wx_Clouds                      int64
wx_Drizzle                     int64
wx_Fog                         int64
wx_Haze                        int64
wx_Mist                        int64
wx_Rain                        int64
wx_Smoke                       int64
wx_Snow                        int64
wx_Squall                      int64
wx_Thunderstorm                int64
is_severe_weather              int64
temp_scaled     

### Step 3: Build the shared feature matrix and both targets

The same 23 engineered features feed both models in this task, only the target differs: `high_risk` for the classifier, `traffic_volume` for the regressor. Building both target vectors alongside a single `FEATURE_COLS` list keeps the two tasks on an identical, leakage-free feature set.

In [8]:
FEATURE_COLS = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "is_weekend", "is_holiday",
    "wx_Clear", "wx_Clouds", "wx_Drizzle", "wx_Fog", "wx_Haze", "wx_Mist",
    "wx_Rain", "wx_Smoke", "wx_Snow", "wx_Squall", "wx_Thunderstorm",
    "is_severe_weather", "is_low_visibility",
    "temp_scaled", "rain_1h", "snow_1h", "clouds_all",
]

X_train, X_test = train_df[FEATURE_COLS], test_df[FEATURE_COLS]

y_train_clf, y_test_clf = train_df["high_risk"], test_df["high_risk"]
y_train_reg, y_test_reg = train_df["traffic_volume"], test_df["traffic_volume"]

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Missing values in X_train: {X_train.isna().sum().sum()}")
print(f"Missing values in X_test:  {X_test.isna().sum().sum()}")
print(f"\ny_train_clf positive rate: {y_train_clf.mean():.2%}")
print(f"y_test_clf  positive rate: {y_test_clf.mean():.2%}")
print(f"\ny_train_reg mean: {y_train_reg.mean():.1f}   y_test_reg mean: {y_test_reg.mean():.1f}")

X_train: (32460, 23)   X_test: (8115, 23)
Missing values in X_train: 0
Missing values in X_test:  0

y_train_clf positive rate: 4.59%
y_test_clf  positive rate: 5.37%

y_train_reg mean: 3285.9   y_test_reg mean: 3309.6


### Step 4: Fit and evaluate baseline classification models

Two algorithms on the same feature set: logistic regression as the linear baseline, and a random forest as the tree-based ensemble, satisfying the brief's requirement for at least two algorithms.

`high_risk` is a rare event, roughly 5% positive in both splits, so both models use `class_weight="balanced"`. Without it, a classifier could predict "not high risk" for every single row and still score about 95% accuracy while being completely useless, which is exactly why accuracy alone is not reported as the headline metric here. Precision, recall, F1, and ROC AUC are evaluated together instead, since precision and recall trade off directly against each other under class imbalance and neither alone tells the full story.

In [16]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train, y_train_clf)

rf_clf = RandomForestClassifier(
    n_estimators=150, max_depth=15, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf_clf.fit(X_train, y_train_clf)

models_clf = {"Logistic Regression": log_reg, "Random Forest": rf_clf}
results_clf = []

for name, model in models_clf.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results_clf.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_clf, y_pred),
        "Precision": precision_score(y_test_clf, y_pred),
        "Recall": recall_score(y_test_clf, y_pred),
        "F1": f1_score(y_test_clf, y_pred),
        "ROC AUC": roc_auc_score(y_test_clf, y_proba),
    })

results_clf_df = pd.DataFrame(results_clf).set_index("Model")
print(results_clf_df.round(4))

                     Accuracy  Precision  Recall      F1  ROC AUC
Model                                                            
Logistic Regression    0.9784     0.7136     1.0  0.8329   0.9978
Random Forest          0.9890     0.8305     1.0  0.9074   0.9988


### Step 5: Fit and evaluate baseline regression models

Same feature set, same train/test split, new target: `traffic_volume`. Linear regression as the baseline, random forest regressor as the tree-based ensemble, evaluated with MAE (mean absolute error, in the same units as traffic volume, easy to interpret directly) and R-squared (the proportion of variance explained).

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train_reg)

rf_reg = RandomForestRegressor(
    n_estimators=150, max_depth=15, min_samples_leaf=10,
    random_state=42, n_jobs=-1,
)
rf_reg.fit(X_train, y_train_reg)

models_reg = {"Linear Regression": lin_reg, "Random Forest": rf_reg}
results_reg = []

for name, model in models_reg.items():
    y_pred = model.predict(X_test)
    results_reg.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_reg, y_pred),
        "R2": r2_score(y_test_reg, y_pred),
    })

results_reg_df = pd.DataFrame(results_reg).set_index("Model")
print(results_reg_df.round(4))

                        MAE      R2
Model                              
Linear Regression  828.2493  0.7089
Random Forest      268.7806  0.9411


### Step 6: Feature importance from the random forest models

Random forest exposes `feature_importances_` directly (mean decrease in impurity across all trees), giving a quick, model-native view of which features drive each prediction. This is a useful sanity check on its own, does the ranking match domain intuition, and it foreshadows the SHAP-based explainability required later in Task 3.

In [18]:
importances_clf = pd.Series(rf_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
importances_reg = pd.Series(rf_reg.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print("Top 8 features — high_risk classifier:")
print(importances_clf.head(8).round(4))

print("\nTop 8 features — traffic_volume regressor:")
print(importances_reg.head(8).round(4))

Top 8 features — high_risk classifier:
is_low_visibility    0.3720
hour_cos             0.1271
wx_Mist              0.1213
wx_Clouds            0.0886
wx_Clear             0.0621
wx_Haze              0.0551
is_severe_weather    0.0408
wx_Thunderstorm      0.0357
dtype: float64

Top 8 features — traffic_volume regressor:
hour_cos       0.6917
hour_sin       0.1814
dow_sin        0.0531
is_weekend     0.0499
temp_scaled    0.0129
dow_cos        0.0064
clouds_all     0.0023
wx_Clouds      0.0007
dtype: float64
